In [66]:
#Importing necessary libraries for data handling, modeling, handling class imbalance, and evaluating.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

#Loading the training and testing datasets.
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv")
y_test = pd.read_csv("data/y_test.csv")

#Converting y_train and y_test to 1D arrays for modeling.
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [67]:
#Getting the unique action classes from the training labels.
actions = np.sort(np.unique(y_train))

#Printing the action classes and the number of classes.
print(actions)
print(f'The number of unique classes is {len(actions)}.')

['appear' 'click' 'disappear' 'drag' 'hover' 'scrolldown' 'scrollup'
 'select' 'type' 'zoomin' 'zoomout']
The number of unique classes is 11.


In [68]:
#Defining the rare classes.
rare_classes = ["type", "hover", "scrolldown", "appear", "disappear", "scrollup"]

#Replacing rare classes with "other" in the training labels.
y_train = pd.Series(y_train).replace(rare_classes, "other").values

#Replacing rare classes with "other" in the testing labels.
y_test = pd.Series(y_test).replace(rare_classes, "other").values

#Getting the unique action classes from the training labels.
actions = np.sort(np.unique(y_train))

In [69]:
#Storing the trained models and evaluation metrics for each class.
ova_models = {}
ova_results = {}

#Looping through each action class to build an ova model.
for action in actions:
    
    #Binary training labels where the current class is 1, all other are 0.
    y_train_binary = (y_train == action).astype(int)
    
    #Binary t labels where the current class is 1, all other are 0.
    y_test_binary = (y_test == action).astype(int)
    
    #Printing the class currently being modeled and the class balance before SMOTE.
    #print(f'For class {action}, the balance before SMOTE is: ')
    #print(pd.Series(y_train_binary).value_counts())

    #Applying SMOTE to balance the binary training data.
    smote = SMOTE(random_state = 42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train_binary)
    
    #Scaling the features.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_resampled)
    X_test_scaled = scaler.transform(X_test)
    
    #Training a logistic regression model on the resampled and scaled data.
    model = LogisticRegression(random_state = 42, max_iter = 1000)
    model.fit(X_train_scaled, y_train_resampled)
    
    #Storing the trained model for the current class.
    ova_models[action] = model

    #Making predictions on the testing data for the current class.
    y_probs = model.predict_proba(X_test_scaled)[:,1]
    y_pred = (y_probs > 0.78).astype(int)
    
    #Calculating evaluation metrics for the current class.
    accuracy = accuracy_score(y_test_binary, y_pred)
    precision = precision_score(y_test_binary, y_pred)
    recall = recall_score(y_test_binary, y_pred)
    f1 = f1_score(y_test_binary, y_pred)
    
    #Storing the evaluation metrics for the current class.
    ova_results[action] = {"Accuracy": accuracy, "Precision": precision,
    "Recall": recall, "F1": f1}
    
#Converting the stored results into a fresh DataFrame.
results_df = pd.DataFrame(ova_results).T

#Printing the fresh results DataFrame.
print(results_df)

         Accuracy  Precision    Recall        F1
click    0.738016   0.708955  0.326460  0.447059
drag     0.615385   0.829268  0.239437  0.371585
other    0.930881   0.191176  0.650000  0.295455
select   0.928651   0.163934  0.434783  0.238095
zoomin   0.960981   0.710843  0.842857  0.771242
zoomout  0.972129   0.800000  0.835821  0.817518
